
# Notebook 03 — Data Preprocessing

## 🎯 Objective

This notebook prepares the dataset for machine learning by performing:

- Data Cleaning
- Missing Value Treatment
- Datetime Feature Extraction
- Encoding
- Feature Scaling
- Train-Test Split
- Class Imbalance Handling (SMOTE)

The processed dataset will be saved for model training.

In [1]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE

print("Libraries Imported Successfully ✅")

Libraries Imported Successfully ✅


In [2]:
# ==========================================================
# Load Dataset
# ==========================================================

DATA_PATH="../data/raw/bank_customer_churn.csv"

df=pd.read_csv(DATA_PATH)

print(df.shape)

df.head()

(28382, 21)


,customer_id,vintage,age,gender,dependents,occupation,city,customer_nw_category,branch_code,current_balance,...,average_monthly_balance_prevQ,average_monthly_balance_prevQ2,current_month_credit,previous_month_credit,current_month_debit,previous_month_debit,current_month_balance,previous_month_balance,churn,last_transaction
0,1,2101,66,Male,0.0,self_employed,187.0,2,755,1458.71,...,1458.71,1449.07,0.20,0.20,0.20,0.20,1458.71,1458.71,0,2019-05-21
1,2,2348,35,Male,0.0,self_employed,NaN,2,3214,5390.37,...,7799.26,12419.41,0.56,0.56,5486.27,100.56,6496.78,8787.61,0,2019-11-01
2,4,2194,31,Male,0.0,salaried,146.0,2,41,3913.16,...,4910.17,2815.94,0.61,0.61,6046.73,259.23,5006.28,5070.14,0,NaT
3,5,2329,90,NaN,NaN,self_employed,1020.0,2,582,2291.91,...,2084.54,1006.54,0.47,0.47,0.47,2143.33,2291.91,1669.79,1,2019-08-06
4,6,1579,42,Male,2.0,self_employed,1494.0,3,388,927.72,...,1643.31,1871.12,0.33,714.61,588.62,1538.06,1157.15,1677.16,1,2019-11-03


In [3]:
# ==========================================================
# Create Working Copy
# ==========================================================

data=df.copy()

print("Working copy created ✅")

Working copy created ✅


In [4]:
# ==========================================================
# Drop ID Columns
# ==========================================================

drop_columns=[]

if "customer_id" in data.columns:
    drop_columns.append("customer_id")

data.drop(columns=drop_columns,inplace=True)

print("Dropped Columns :",drop_columns)

Dropped Columns : ['customer_id']


In [5]:
# ==========================================================
# Datetime Conversion
# ==========================================================

data["last_transaction"]=pd.to_datetime(
    data["last_transaction"],
    errors="coerce"
)

data["transaction_year"]=data["last_transaction"].dt.year
data["transaction_month"]=data["last_transaction"].dt.month
data["transaction_day"]=data["last_transaction"].dt.day
data["transaction_weekday"]=data["last_transaction"].dt.day_name()

data.drop(columns=["last_transaction"],inplace=True)

data.head()

,vintage,age,gender,dependents,occupation,city,customer_nw_category,branch_code,current_balance,previous_month_end_balance,...,previous_month_credit,current_month_debit,previous_month_debit,current_month_balance,previous_month_balance,churn,transaction_year,transaction_month,transaction_day,transaction_weekday
0,2101,66,Male,0.0,self_employed,187.0,2,755,1458.71,1458.71,...,0.20,0.20,0.20,1458.71,1458.71,0,2019.0,5.0,21.0,Tuesday
1,2348,35,Male,0.0,self_employed,NaN,2,3214,5390.37,8704.66,...,0.56,5486.27,100.56,6496.78,8787.61,0,2019.0,11.0,1.0,Friday
2,2194,31,Male,0.0,salaried,146.0,2,41,3913.16,5815.29,...,0.61,6046.73,259.23,5006.28,5070.14,0,NaN,NaN,NaN,NaN
3,2329,90,NaN,NaN,self_employed,1020.0,2,582,2291.91,2291.91,...,0.47,0.47,2143.33,2291.91,1669.79,1,2019.0,8.0,6.0,Tuesday
4,1579,42,Male,2.0,self_employed,1494.0,3,388,927.72,1401.72,...,714.61,588.62,1538.06,1157.15,1677.16,1,2019.0,11.0,3.0,Sunday


In [6]:
# ==========================================================
# Missing Values
# ==========================================================

missing=data.isnull().sum()

display(
    missing[missing>0]
)

gender                  525
dependents             2463
occupation               80
city                    803
transaction_year       3223
transaction_month      3223
transaction_day        3223
transaction_weekday    3223
dtype: int64

In [7]:
# ==========================================================
# Target Column
# ==========================================================

target="churn"

X=data.drop(columns=[target])

y=data[target]

print(X.shape)
print(y.shape)

(28382, 22)
(28382,)


In [8]:
# ==========================================================
# Feature Types
# ==========================================================

numeric_features=X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features=X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numeric :",len(numeric_features))
print("Categorical :",len(categorical_features))

Numeric : 19
Categorical : 3


In [9]:
# ==========================================================
# Numeric Pipeline
# ==========================================================

numeric_pipeline=Pipeline(

    [

        ("imputer",
         SimpleImputer(strategy="median")),

        ("scaler",
         StandardScaler())

    ]

)

In [10]:
# ==========================================================
# Categorical Pipeline
# ==========================================================

categorical_pipeline=Pipeline(

    [

        ("imputer",
         SimpleImputer(strategy="most_frequent")),

        ("encoder",
         OneHotEncoder(handle_unknown="ignore"))

    ]

)

In [11]:
# ==========================================================
# Column Transformer
# ==========================================================

preprocessor=ColumnTransformer(

    [

        ("num",
         numeric_pipeline,
         numeric_features),

        ("cat",
         categorical_pipeline,
         categorical_features)

    ]

)

In [12]:
# ==========================================================
# Transform Dataset
# ==========================================================

X_processed=preprocessor.fit_transform(X)

print(X_processed.shape)

(28382, 33)


In [13]:
# ==========================================================
# Train Test Split
# ==========================================================

X_train,X_test,y_train,y_test=train_test_split(

    X_processed,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

print(X_train.shape)
print(X_test.shape)

(22705, 33)
(5677, 33)


In [14]:
# ==========================================================
# Handle Imbalanced Dataset
# ==========================================================

smote=SMOTE(random_state=42)

X_train_smote,y_train_smote=smote.fit_resample(

    X_train,

    y_train

)

print("Before SMOTE")

print(y_train.value_counts())

print()

print("After SMOTE")

print(pd.Series(y_train_smote).value_counts())

Before SMOTE
churn
0    18497
1     4208
Name: count, dtype: int64

After SMOTE
churn
0    18497
1    18497
Name: count, dtype: int64


In [15]:
# ==========================================================
# Save Preprocessor
# ==========================================================

Path("../models").mkdir(exist_ok=True)

joblib.dump(

    preprocessor,

    "../models/preprocessor.pkl"

)

print("Preprocessor Saved ✅")

Preprocessor Saved ✅


In [16]:
# ==========================================================
# Save Processed Dataset
# ==========================================================

processed=pd.DataFrame(

    X_train_smote

)

processed["churn"]=y_train_smote

Path("../data/processed").mkdir(exist_ok=True)

processed.to_csv(

    "../data/processed/processed_dataset.csv",

    index=False

)

print("Processed Dataset Saved ✅")

Processed Dataset Saved ✅


In [17]:
# ==========================================================
# Verification
# ==========================================================

print("Processed Shape :",processed.shape)

processed.head()

Processed Shape : (36994, 34)


,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,30,31,32,churn
0,0.252523,-0.966389,0.712562,0.700230,1.172672,-0.475563,-0.133784,-0.130690,-0.135197,-0.126553,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
1,-0.110551,-0.460965,-0.330877,1.705519,1.172672,-0.278289,-0.133200,-0.136127,-0.138771,-0.126228,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0
2,0.036146,-0.067858,-0.330877,-1.085387,-0.341489,-0.268692,-0.104552,-0.115572,-0.114066,-0.092763,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
3,1.011677,2.346943,-0.330877,0.899413,-1.855650,-0.789069,0.149746,0.148655,0.152553,0.127798,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,-1.555510,-0.011700,-0.330877,-1.832910,-0.341489,-0.040494,-0.021270,-0.024014,-0.024500,-0.061926,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0


# 📌 Summary

## Completed

- Removed unnecessary columns
- Converted datetime features
- Handled missing values
- Encoded categorical variables
- Standardized numeric features
- Split dataset into train and test sets
- Applied SMOTE to balance classes
- Saved preprocessing pipeline
- Saved processed dataset

The dataset is now ready for Feature Engineering and Model Training.